In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from trl import SFTTrainer

In [ ]:
model_id = "unsloth/Llama-3.2-3B-Instruct"


# Tokenizer and padding configuring

# We load this first because we need the EOS token for the dataset
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Load and format dataset

dataset = load_dataset("ServiceNow-AI/R1-Distill-SFT", 'v0', split="train")
EOS_TOKEN = tokenizer.eos_token

r1_prompt = """You are a reflective assistant engaging in thorough, iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer.
<problem>
{}
</problem>
{}
{}
"""

def formatting_prompts_func(examples):  # Hugging Face dataset.map(...) function creates examples automatically where it is a batch of rows from the datset
    problems = examples["problem"]
    thoughts = examples["reannotated_assistant_content"]
    solutions = examples["solution"]
    texts = []
    for problem, thought, solution in zip(problems, thoughts, solutions):
        # Inject data into prompt and append the crucial stop token
        text = r1_prompt.format(problem, thought, solution) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Apply formatting and create the "text" column for the trainer
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.0 MB/s eta 0:00:00


In [ ]:


bnb_config = BitsAndBytesConfig( # bnb config, does not load model, just defines it
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config = bnb_config,
    device_map = "auto",
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [ ]:
# Wrap in PEFT / LoRA adapters

model.gradient_checkpointing_enable() #reduces ram but increases compute time
model = prepare_model_for_kbit_training(model) # Training prep step. Freezes base model layers and prepares PEFT adapters

peft_config = LoraConfig(
    r = 16,
    lora_alpha = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj" # Older AI models just had two layers in their MLP (one to expand the data, one to shrink it). But Llama uses a newer, smarter design called a SwiGLU MLP
    ],
    bias = "none",
    task_type = "CAUSAL_LM"
)

# inject LoRA adapters onto the target layers
model = get_peft_model(model, peft_config)
model.print_trainable_parameters() # see adapter reduction

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [ ]:
from trl import SFTConfig

In [ ]:
trainer = SFTTrainer(   # trainer object that will run SFT
    model = model,
    train_dataset = dataset,
    processing_class = tokenizer,


    args = SFTConfig(
        dataset_text_field = "text",
        max_length = 2048,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 50,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
) # around 3 mins per step. total 3 hours, Colab crashed before that. So I changed steps to 45 and 50

trainer.train()

Step,Training Loss
1,0.808024
2,0.700717
3,0.736907
4,0.704997
5,0.608012
6,0.655255
7,0.556939
8,0.609261
9,0.655200
10,0.618470


Step,Training Loss
1,0.808024
2,0.700717
3,0.736907
4,0.704997
5,0.608012
6,0.655255
7,0.556939
8,0.609261
9,0.655200
10,0.618470


TrainOutput(global_step=50, training_loss=0.5565608602762222, metrics={'train_runtime': 7938.8827, 'train_samples_per_second': 0.05, 'train_steps_per_second': 0.006, 'total_flos': 1.2637998604099584e+16, 'train_loss': 0.5565608602762222})

In [ ]:
# save and download lora adapters

trainer.model.save_pretrained("r1_lora_adapters")
tokenizer.save_pretrained("r1_lora_adapters")
print("Training complete and adapters saved!")

Training complete and adapters saved!


In [ ]:

import shutil
from google.colab import files


shutil.make_archive("r1_LoRA_adapters", 'zip', "r1_lora_adapters")
files.download("r1_LoRA_adapters.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#INFERENCE

In [ ]:
!unzip r1_LoRA_adapters.zip -d r1_lora_adapters

Archive:  r1_LoRA_adapters.zip
  inflating: r1_lora_adapters/tokenizer_config.json  
  inflating: r1_lora_adapters/chat_template.jinja  
  inflating: r1_lora_adapters/tokenizer.json  
  inflating: r1_lora_adapters/README.md  
  inflating: r1_lora_adapters/adapter_config.json  
  inflating: r1_lora_adapters/adapter_model.safetensors  


In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

In [ ]:
base_model_id = "unsloth/Llama-3.2-3B-Instruct"
adapter_path = "r1_lora_adapters"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
# LOAD BASE MODEL IN 4-BIT

# We load the base model with the exact same 4-bit quantization we trained it with
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.bfloat16
)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config = bnb_config,
    device_map = "auto",
)

Loading base model...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [ ]:
# ATTACH THE LORA ADAPTERS

print("Attaching LoRA adapters...")
model = PeftModel.from_pretrained(base_model, adapter_path)

Attaching LoRA adapters...


In [ ]:
# FORMAT THE PROMPT


r1_prompt = """You are a reflective assistant engaging in thorough, iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer.
<problem>
{}
</problem>

"""

question = "What is the capital of Pakistan and why is it historically significant?"
formatted_prompt = r1_prompt.format(question)

In [ ]:
# GENERATE THE RESPONSE
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

print("Generating stream-of-consciousness response...")
outputs = model.generate(
    **inputs,
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 1.5,
    min_p = 0.1
)

# Decode the raw numbers and print
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)

Generating stream-of-consciousness response...


In [ ]:
print("\n*** OUTPUT ***\n")
print(response[0])


*** OUTPUT ***

You are a reflective assistant engaging in thorough, iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer.
<problem>
What is the capital of Pakistan and why is it historically significant?
</problem>

<think>
Okay, so this question is about the capital of Pakistan. I remember learning about different countries, including Pakistan, in my geography or social studies class. Now, the question is asking me to identify the capital and provide a reason why it's significant. Hmm, let's try to break this down.

First, I recall that Pakistan is located in South Asia and shares borders with countries like India, China, Afghanistan, Iran, and others. It's an independent country with its own government. Now, thinking about capitals, I know that each country has its own capital city. The capital is often the seat of government, the location of the head office